# `treasury(start, end=None, tenor=None)`

Daily US Treasury par yields by tenor, in percent, from FMP via `macro.treasury_rates`. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

In [ ]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import treasury
from arora_data.treasury import TENORS, TreasuryRate


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The tenors
Twelve, from 1 month to 30 years, each stored as a column of `macro.treasury_rates`. A tenor works by its code in any case and spacing (`"10Y"`, `"10y"`, `"10 y"`) or by its column name (`"year10"`).

In [ ]:
pl.DataFrame({"tenor": list(TENORS), "column": list(TENORS.values())})

## One tenor
`tenor="10Y"`: the 10-year yield by date. `yield_pct` is in percent, so 4.86 is 4.86%.

In [ ]:
week = today_et() - timedelta(days=7)
assert treasury(week, tenor="10Y") == treasury(week, tenor="10y") == treasury(week, tenor="year10")
show(treasury(week, tenor="10Y"))

## Every tenor, the default
Leave `tenor` out (or pass `"ALL"`) for the whole curve, ordered by maturity then date. Pivot it for one row per day.

In [ ]:
curve = to_df(treasury(today_et() - timedelta(days=7)), TreasuryRate)
curve.pivot(on="tenor", index="date", values="yield_pct")

## The curve on one day, and a year earlier

In [ ]:
last = curve["date"].max()
year_ago = to_df(treasury(last - timedelta(days=372), last - timedelta(days=365)), TreasuryRate)
year_ago_day = year_ago["date"].max()
(
    curve.filter(pl.col("date") == last).select("tenor", pl.col("yield_pct").alias(str(last)))
    .join(year_ago.filter(pl.col("date") == year_ago_day).select("tenor", pl.col("yield_pct").alias(str(year_ago_day))),
          on="tenor", how="left", maintain_order="left")
)

## Spreads come from two tenors
There is no spread argument: join two tenors. Here the 10-year minus the 2-year, the usual recession signal when it turns negative, over the last five years.

In [ ]:
start = today_et() - timedelta(days=5 * 365)
ten, two = (to_df(treasury(start, tenor=t), TreasuryRate).select("date", pl.col("yield_pct").alias(t)) for t in ("10Y", "2Y"))
spread = ten.join(two, on="date").with_columns((pl.col("10Y") - pl.col("2Y")).alias("10Y-2Y"))
print(f"{spread.filter(pl.col('10Y-2Y') < 0).height} of {spread.height} days inverted")
spread.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True).agg(
    pl.col("10Y-2Y").min().alias("min"), pl.col("10Y-2Y").max().alias("max"), pl.col("10Y-2Y").last().alias("last"))

## Full history in one call
From 1990, the table's first day. There is no row limit. Each tenor's first day follows Treasury's own history: the 20-year from 1993, the 1-month from July 2001 (bar the 2001-03-31 placeholder, below), the 2-month from 2018.

In [ ]:
history = to_df(treasury(date(1900, 1, 1)), TreasuryRate)
print(f"{history.height:,} rows over {history['date'].n_unique():,} days, {history['date'].min()} to {history['date'].max()}")
history.group_by("tenor", maintain_order=True).agg(pl.col("date").min().alias("first"), pl.col("date").max().alias("last"), pl.len().alias("days"))

## Gaps are left out, not filled
A tenor with no value that day has no row. The 30-year was suspended from 2002-02-15 to 2006-02-09, and Columbus Day 2010 (10-11) is stored with no values at all, so it never appears.

In [ ]:
thirty = history.filter(pl.col("tenor") == "30Y").with_columns(pl.col("date").shift().alias("previous"))
print(thirty.filter((pl.col("date") - pl.col("previous")).dt.total_days() > 7).select("previous", "date"))
show(treasury(date(2010, 10, 8), date(2010, 10, 12), tenor="10Y"))

## Rows are served as stored
The table's one weekend row, Saturday 2001-03-31, is an FMP placeholder: the curve is inverted (10-year 4.14 against 4.93 the day before) and it holds the only 1-month and 2-month values before July 2001. It comes back unchanged.

In [ ]:
around = to_df(treasury(date(2001, 3, 29), date(2001, 4, 3)), TreasuryRate)
around.pivot(on="tenor", index="date", values="yield_pct").sort("date")

## Bad input
A tenor that isn't one of the twelve, or a start after the end, raises a clear error.

In [ ]:
for tenor in ["4M", "10", "10 years"]:
    try:
        treasury(week, tenor=tenor)
    except ValueError as err:
        print(f"{tenor}: {err}")
try:
    treasury(today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")